# Long processing: baselines over months of backups

Some algorithms need a long history. A baseline (of the CO2 or of the electrochemical sensors) cannot be told from the signal on the 500 readings of a processing run, and requesting months of data from the Smart Citizen API on every run is not possible.

The **backups** are the source: the data of each device in parquet files (in S3, written by `device.backup_to_storage()`). A **long blueprint** (`meta.kind` = `long`, e.g. `sc_air_baseline`) processes a window of them (`meta.window_days`, 90 days). This is what Smart Citizen Flows runs every week for the hardware that lists a long blueprint.

This notebook needs the storage extra:

```
pip install "scdata[storage]"
```

In [ ]:
import logging

import scdata as sc
import smartcitizen_connector
from scdata.tools.custom_logger import set_logger_level

# Only errors: 'WARNING', 'INFO' or 'DEBUG' show what scdata and smartcitizen-connector do
set_logger_level('ERROR')
logging.getLogger('smartcitizen_connector').setLevel('ERROR')

sc.__version__

## Backups

`device.backup_to_storage()` appends the data of a device to `s3://<S3_DATA_BUCKET>/devices/<id>/data/` (it needs `S3_DATA_BUCKET` and AWS credentials), and `device.load_from_storage()` reads it back.

To run this notebook without S3, we write a backup of the last two weeks to a local folder with the same layout. `load_from_storage(root=...)` reads from a local folder instead of S3.

In [ ]:
import os
import pandas as pd
from os.path import join
from scdata._config import config

root = join(config.paths['data'], 'storage_example')

source = sc.Device(params=sc.APIParams(id=18154))
source.options.min_date = (pd.Timestamp.now(tz='UTC') - pd.Timedelta('14D')).strftime('%Y-%m-%d')
await source.load()

# Same layout as backup_to_storage: devices/<id>/data/, with a TIME column
folder = join(root, 'devices', str(source.id), 'data')
os.makedirs(folder, exist_ok=True)
source.data.assign(TIME=source.data.index).to_parquet(join(folder, 'backup.parquet'), index=False)
source.data.shape

In [ ]:
# With S3 (S3_DATA_BUCKET and AWS credentials set), the same backup goes to the bucket
if 'S3_DATA_BUCKET' in os.environ:
    print('Backing up to', os.environ['S3_DATA_BUCKET'])
    # source.backup_to_storage(path='devices')

## Read a period of the backups

`load_from_storage` reads only a period (`min_date`, `max_date`) and some columns (`channels`): the filter is pushed down to the parquet files, so only what is needed is read. Without `root`, it reads from `s3://<S3_DATA_BUCKET>`.

In [ ]:
device = sc.Device(params=sc.APIParams(id=source.id))
start = (pd.Timestamp.now(tz='UTC') - pd.Timedelta('10D')).strftime('%Y-%m-%d')

device.load_from_storage(min_date=start, channels=['TEMP', 'HUM', 'NOISE_A'], root=root)
device.data.index.min(), device.data.index.max(), list(device.data.columns)

## A long blueprint

A device loaded with `blueprint_kind='long'` uses the long blueprint of its hardware: `sc.Device(params=sc.APIParams(id=...), blueprint_kind='long')`. To try another long blueprint, read it from the same `BASE_POSTPROCESSING_URL` (long blueprints are not loaded with the others in `config.blueprints`):

In [ ]:
import requests

name = 'sc_air_baseline'
blueprint = requests.get(f'{config._base_postprocessing_url}blueprints/{name}.json', timeout=30).json()
blueprint['meta']

In [ ]:
[(channel['name'], channel['function']) for channel in blueprint['channels']]

`device.use_blueprint()` processes the device with another blueprint, filled with the sensors of each version of its hardware (as the device's own blueprint is). It returns `True` if the channels were filled from the hardware.

In [ ]:
device.use_blueprint(name, blueprint)

In [ ]:
# The electrochemical channels use alphasense_als: the 803 algorithm with a baseline of the working electrode
next(channel for channel in device.channels if channel.name == 'NO2').kwargs

Load only the sensors the blueprint needs, for its window. Here the window is the two weeks of our backup; flows uses `window_days` (90 days) from the first day of that month.

In [ ]:
window_days = min(blueprint['meta']['window_days'], 14)
start = (pd.Timestamp.now(tz='UTC') - pd.Timedelta(f'{window_days}D')).strftime('%Y-%m-%d')

device.load_from_storage(min_date=start, channels=device.required_sensors, root=root)
device.data.shape

In [ ]:
device.process()

In [ ]:
device.data[['NO2', 'O3']].describe().T

Compare with the short processing (`sc_air`, the 803 algorithm without baseline) on the same data:

In [ ]:
short = sc.Device(params=sc.APIParams(id=source.id))
short.load_from_storage(min_date=start, channels=short.required_sensors, root=root)
short.process()

pd.DataFrame({'sc_air (803)': short.data[['NO2', 'O3']].median(),
              f'{name} (ALS)': device.data[['NO2', 'O3']].median()}).round(1)

## Tune the parameters

The baseline parameters (`lam`, `p`, ...) are starting values that work for most sensors of the same type. Change them in the channel and process again. Smart Citizen Flows keeps them per sensor type (parameter sets), and per hardware.

In [ ]:
no2 = next(channel for channel in device.channels if channel.name == 'NO2')
results = {}
for lam in [1e8, 1e10]:
    no2.kwargs['lam'] = lam
    device.process(channels_list=['NO2'])
    results[f'lam={lam:.0e}'] = device.data['NO2'].describe()
pd.DataFrame(results).round(1)

## Health checks and results

Long blueprints have health checks too (see [20_health_checks.ipynb](20_health_checks.ipynb)). The results are stored, not posted to the Smart Citizen API: flows writes them per month to `devices/<id>/processed/<blueprint>/` in S3. Here, an export:

In [ ]:
device.health_checks()
[(check['name'], check['status']) for check in device.health['checks']]

In [ ]:
channels = [channel.name for channel in device.channels if channel.name in device.data.columns]
device.data[channels].to_parquet(join(root, 'devices', str(device.id), f'{name}.parquet'))
channels